In [1]:
# ---- 3-panel Taylor on 20-yr (1986-2005) climatologies, land + ocean buoys ----
# Exactly the taylor_3panel figure, but each station is reduced to its
# 20-year annual-mean climatology before pooling (T / PR / wind panels).
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import taylor_figure as tf
warnings.filterwarnings('ignore')
BASE = os.getcwd()
with open(os.path.join(BASE, 'pickles', 'taylor_annuals.pkl'), 'rb') as f:
    TA = pickle.load(f)
print('taylor cache version:', TA['_version'])
MODELS = tf.MODELS
def climo(a):
    a = np.asarray(a, float); a = a[np.isfinite(a)]
    return a.mean() if a.size else np.nan
# build (obs_climo, models_climo) per var over the FULL 3-panel station set
DATA = {}
for v in ('t', 'pr', 'wind'):
    obs, models = TA[v]
    o = {s: climo(obs[s]) for s in obs}
    m = {mm: {s: climo(models[mm][s]) for s in models[mm]} for mm in MODELS}
    DATA[v] = (o, m)
    print(v, 'obs stations =', len(o))


taylor cache version: 10
t obs stations = 164
pr obs stations = 172
wind obs stations = 23


In [2]:
# Taylor inputs on climatology values (per model over its shared stations)
def climo_refstd(obs, models):
    keep = set()
    for m in MODELS:
        keep |= set(obs) & set(models.get(m, {}))
    vals = [obs[s] for s in sorted(keep) if np.isfinite(obs[s])]
    return float(np.std(vals))
def climo_stats(obs, models, label):
    m = models.get(label, {})
    sh = [s for s in sorted(set(obs) & set(m))
          if np.isfinite(obs[s]) and np.isfinite(m[s])]
    if len(sh) < 3:
        return np.nan, np.nan, np.nan, len(sh)
    o = np.array([obs[s] for s in sh]); mm = np.array([m[s] for s in sh])
    return float(np.std(mm)), float(np.corrcoef(o, mm)[0, 1]), None, len(sh)
print('%-4s %-8s %5s %9s %9s %7s' % ('var', 'model', 'n', 'std', 'refstd', 'r'))
for v in ('t', 'pr', 'wind'):
    o, m = DATA[v]
    ref = climo_refstd(o, m)
    for lab in MODELS:
        std, corr, _, n = climo_stats(o, m, lab)
        print('%-4s %-8s %5d %9.3f %9.3f %7.3f' % (v, lab, n, std, ref, corr))


var  model        n       std    refstd       r
t    D03        154     2.442     2.130   0.885
t    D02        154     2.606     2.130   0.783
t    D01        154     2.476     2.130   0.684
t    CanESM2    154     2.420     2.130   0.669
t    CanRCM4    154     3.297     2.130   0.757
pr   D03        159   710.943   882.315   0.944
pr   D02        159  1246.435   882.315   0.925
pr   D01        159  1015.516   882.315   0.830
pr   CanESM2    159   315.550   882.315   0.333
pr   CanRCM4    159   820.430   882.315   0.830
wind D03         23     1.894     2.093   0.963
wind D02         23     1.907     2.093   0.940
wind D01         23     1.970     2.093   0.846
wind CanESM2     23     1.521     2.093   0.759
wind CanRCM4     23     1.992     2.093   0.894


In [3]:
# ---- Draw the 3-panel figure (mirrors tf.make_figure layout, climatology data) ----
FIG_W = tf.FIG_W_MM / 25.4
FIG_H = tf.FIG_H
def draw_panel(fig, rect, data, var, refstd, max_std, levels):
    obs, models = data
    stats = {}
    for mm in MODELS:
        std, corr, _, n = climo_stats(obs, models, mm)
        stats[mm] = (std, corr, n)
    dia = tf.TaylorDiagram(refstd, fig=fig, rect=rect, label='Observations',
                           max_std=max_std, unit=tf.VAR_UNITS[var])
    dia.put_ticks_out()
    ct = dia.add_contours(levels, colors='0.55', linewidths=0.9, zorder=0)
    # label_contours(corr=0.55) labels each level where it crosses the
    # shared corr=0.55 diagonal but silently skips levels whose contour
    # never reaches that diagonal.  In the wind panel (refstd 2.09,
    # axis to 2.5) even the top contour only spans corr in
    # [0.087, 0.996], so the shared diagonal misses EVERY level and the
    # panel is left unlabeled.  Fallback: label where each contour
    # crosses the mid-radius ring r = (smin+smax)/2 (guaranteed for any
    # level within [refstd - r_mid, refstd + r_mid]); a level too close
    # to the refstd star to intersect the ring (disc < 0 there) is
    # placed at the contour's own maximum-RMS point (corr = 0) instead.
    # White halo keeps the labels legible on the grey lines.
    if not dia.label_contours(ct, corr=0.55):
        # label_contours() re-clamps the angle with
        # a = max(a, arcsin(l/refstd) + 0.08), so a per-level
        # corr arg does not reliably place the label on the
        # mid-radius ring.  Place each label at the point where the
        # contour crosses r = r_mid (the root closest to mid-radius),
        # or, for a level too close to the refstd star to reach the
        # ring, at the contour's maximum-RMS point (corr = 0).
        r_mid = 0.5 * (dia.smin + dia.smax)
        # 1 decimal: the step is 0.5, and the integer format would
        # label the 0.5 contour "0" (ambiguous with no zero contour).
        fmt = '%.1f'
        for lvl in ct.levels:
            arg = (dia.refstd ** 2 + r_mid ** 2 - lvl ** 2) / (
                2.0 * dia.refstd * r_mid)
            a = np.arccos(np.clip(arg, -1.0, 1.0))
            b = -2.0 * dia.refstd * np.cos(a)
            disc = b * b - 4.0 * (dia.refstd ** 2 - lvl ** 2)
            if disc < 0:
                continue
            roots = [(-b - np.sqrt(disc)) / 2.0,
                     (-b + np.sqrt(disc)) / 2.0]
            roots = [r for r in roots
                     if dia.smin + 1e-9 <= r <= dia.smax - 1e-9]
            if not roots:
                continue
            rr = min(roots, key=lambda r: abs(r - r_mid))
            dia.ax.text(a, rr, fmt % lvl, ha='center', va='center',
                        fontsize=tf.FONT_S, color='0.35', zorder=500,
                        bbox=dict(boxstyle='round,pad=0.12', fc='white',
                                  ec='none', alpha=0.8, lw=0))
    for mm in MODELS:
        std, corr, n = stats[mm]
        if not (np.isnan(std) and np.isnan(corr)):
            dia.add_sample(std, corr, marker=tf.MARKERS[mm], ms=7,
                           color=tf.COLORS[mm], mec=tf.COLORS[mm], mew=0.8,
                           zorder=3, label=tf.LABELS[mm])
    dia._cosmetics()

fig = plt.figure(figsize=(FIG_W, FIG_H))
x0 = 0.065; panel_w = 0.235; panel_gap = 0.095
panel_h = panel_w * FIG_W / FIG_H
panel_y = 0.27
rects = [[x0 + i * (panel_w + panel_gap), panel_y, panel_w, panel_h] for i in range(3)]
for i, (var, title, unit) in enumerate(tf.VARS):
    o, m = DATA[var]
    refstd = climo_refstd(o, m)
    fin = [climo_stats(o, m, mm)[0] for mm in MODELS]
    fin = [s for s in fin if not np.isnan(s)]
    max_std, _ = tf._round_axis_ceiling(max([refstd] + fin) * 1.15, n_intervals=5)
    levels = tf._auto_levels(max_std)
    draw_panel(fig, rects[i], DATA[var], var, refstd, max_std, levels)
    fig.text(rects[i][0] + rects[i][2] / 2.0, panel_y + panel_h + 0.135, title,
             ha='center', va='bottom', fontsize=tf.FONT)
handles = [plt.Line2D([], [], marker='*', ls='', ms=12, color='black',
                      mec='black', mew=0.8, label='Observations (climatology)')]
handles += [plt.Line2D([], [], marker=tf.MARKERS[mm], ls='', ms=7,
                       color=tf.COLORS[mm], mec=tf.COLORS[mm], mew=0.8,
                       label=tf.LABELS[mm]) for mm in MODELS]
fig.legend(handles=handles, loc='lower center', ncol=len(handles),
           bbox_to_anchor=(0.5, 0.0), frameon=False, handlelength=1.0,
           handletextpad=0.4, columnspacing=1.2, fontsize=tf.FONT_S)
os.makedirs(os.path.join(BASE, 'figures', 'climate'), exist_ok=True)
OUTF = os.path.join(BASE, 'figures', 'climate', 'climate_all_taylor.png')
fig.savefig(OUTF, dpi=600)
fig.savefig(OUTF.replace('.png', '.pdf'))
print('saved', OUTF)
plt.close(fig)


saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/climate/climate_all_taylor.png
